<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🔬 Chapter 9 — Model Evaluation and Selection</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Use k-fold and stratified k-fold cross-validation
- Tune hyperparameters with GridSearchCV and RandomizedSearchCV
- Understand the bias-variance trade-off quantitatively
- Draw and interpret learning curves
- Implement train/validation/test split correctly to avoid data leakage

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets        import make_classification
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, GridSearchCV,
    RandomizedSearchCV, learning_curve, train_test_split
)
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import f1_score

%matplotlib inline
np.random.seed(42)

# Synthetic classification dataset
X, y = make_classification(
    n_samples=1000, n_features=15, n_informative=8,
    n_redundant=4, random_state=42, class_sep=0.8
)
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features, class balance: {y.mean():.2f}')

## 📖 Section A — Cross-Validation

A single train/test split gives a noisy estimate of performance. **k-fold CV** repeats the split k times:

```
Fold 1: [TEST][train][train][train][train]
Fold 2: [train][TEST][train][train][train]
...
Fold 5: [train][train][train][train][TEST]
```

Final score = mean ± std of all fold scores.

```python
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='f1')
print(f'F1: {scores.mean():.3f} ± {scores.std():.3f}')
```

> 💡 **Key Rule:** Always use `StratifiedKFold` for classification — it preserves class ratios in each fold. Plain `KFold` may create folds with very different class distributions.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Cross-Validation Comparison
# ─────────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
}

print(f'{"Model":22s}  {"Mean F1":>10}  {"Std F1":>10}')
print('-' * 46)
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=cv, scoring='f1')
    print(f'{name:22s}  {scores.mean():10.4f}  {scores.std():10.4f}')

## 📖 Section B — Hyperparameter Tuning

```python
# GridSearchCV — exhaustive (all combinations)
grid = {'max_depth': [3, 5, 10, None], 'n_estimators': [50, 100, 200]}
search = GridSearchCV(RandomForestClassifier(), grid, cv=5, scoring='f1', n_jobs=-1)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best score: ', search.best_score_)

# RandomizedSearchCV — faster, good enough for large spaces
from scipy.stats import randint
param_dist = {'max_depth': [3, 5, 10, None], 'n_estimators': randint(10, 500)}
rs = RandomizedSearchCV(RandomForestClassifier(), param_dist, n_iter=20, cv=5)
```

> 💡 **Key Rule:** Never tune hyperparameters on the test set. Tune on validation set (or via CV on train set only). Evaluate once on the test set at the very end.

---
## 🟢 Exercise 9.1 — 5-Fold CV for All Models *(Level 1 · Est. 10 min)*

> Run 5-fold stratified CV for all four models. Report mean ± std for accuracy AND F1. Create a summary DataFrame.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 9.1: 5-Fold Cross-Validation
# ─────────────────────────────────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# YOUR CODE HERE

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

assert len(results_df) == 4, 'Should have 4 model rows'
assert 'mean_f1' in results_df.columns
print('\n✅ CV comparison complete!')

In [ ]:
# @title ✅ Solution — Exercise 9.1 (click to expand)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, m in models.items():
    acc_scores = cross_val_score(m, X, y, cv=cv, scoring='accuracy')
    f1_scores  = cross_val_score(m, X, y, cv=cv, scoring='f1')
    results.append({
        'model':    name,
        'mean_acc': round(acc_scores.mean(), 4),
        'std_acc':  round(acc_scores.std(),  4),
        'mean_f1':  round(f1_scores.mean(),  4),
        'std_f1':   round(f1_scores.std(),   4),
    })

results_df = pd.DataFrame(results).sort_values('mean_f1', ascending=False)
print(results_df.to_string(index=False))
print('\n✅ Random Forest typically wins on accuracy; Decision Tree has highest variance (std).')

---
## 🟡 Exercise 9.2 — GridSearchCV for Random Forest *(Level 2 · Est. 20 min)*

> Use GridSearchCV to tune Random Forest: n_estimators=[50,100,200], max_depth=[3,5,10,None], min_samples_split=[2,5]. Report best params and improvement over default.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 9.2: GridSearchCV Hyperparameter Tuning
# ─────────────────────────────────────────────────────────────

param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [3, 5, 10, None],
    'min_samples_split': [2, 5],
}

# YOUR CODE HERE
# grid_search = ...

print('Best params:', grid_search.best_params_)
print(f'Best CV F1: {grid_search.best_score_:.4f}')

# Compare with default
default_rf  = RandomForestClassifier(random_state=42)
default_f1  = cross_val_score(default_rf, X, y, cv=cv, scoring='f1').mean()
print(f'Default RF F1: {default_f1:.4f}')
print(f'Improvement:   {(grid_search.best_score_ - default_f1):.4f}')

In [ ]:
# @title ✅ Solution — Exercise 9.2 (click to expand)

param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [3, 5, 10, None],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X, y)

print('Best params:', grid_search.best_params_)
print(f'Best CV F1: {grid_search.best_score_:.4f}')

default_f1 = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=cv, scoring='f1').mean()
print(f'Default RF F1: {default_f1:.4f}')
print(f'Improvement:   +{(grid_search.best_score_ - default_f1):.4f}')
print('\n✅ Even small improvements matter in production. GridSearchCV tests every combination.')

---
## 🔴 Exercise 9.3 — Learning Curves and Bias-Variance Diagnosis *(Level 3 · Est. 25 min)*

> Plot learning curves for Logistic Regression and Decision Tree. Diagnose high bias vs high variance. Suggest a fix for each.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 9.3: Learning Curves
# ─────────────────────────────────────────────────────────────

def plot_learning_curve(model, X, y, title, ax):
    """
    Plot training and validation scores vs training set size.
    High bias: both curves plateau at low score.
    High variance: large gap between train and val curves.
    """
    # YOUR CODE HERE — use sklearn's learning_curve function
    pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_learning_curve(LogisticRegression(max_iter=1000), X, y, 'Logistic Regression', axes[0])
plot_learning_curve(DecisionTreeClassifier(random_state=42), X, y, 'Decision Tree', axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# @title ✅ Solution — Exercise 9.3 (click to expand)

def plot_learning_curve(model, X, y, title, ax):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='f1',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    val_mean   = val_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Train F1')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='steelblue')
    ax.plot(train_sizes, val_mean,   'o-', color='darkorange', label='Val F1')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='darkorange')
    ax.set_title(title)
    ax.set_xlabel('Training set size')
    ax.set_ylabel('F1 Score')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_learning_curve(LogisticRegression(max_iter=1000), X, y, 'Logistic Regression', axes[0])
plot_learning_curve(DecisionTreeClassifier(random_state=42), X, y, 'Decision Tree', axes[1])
plt.suptitle('Learning Curves: Diagnosing Bias vs Variance', fontsize=13)
plt.tight_layout()
plt.show()

print('\nDiagnosis:')
print('Logistic Regression: If both curves plateau below 0.9 → high bias → use more features or complex model')
print('Decision Tree: If big gap between train and val → high variance → add regularisation (max_depth, min_samples_split)')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: Why use cross-validation instead of a single train/test split?</strong></summary>

**Answer:** A single split gives a performance estimate that depends heavily on which data ended up in test. With small datasets, one 'easy' test fold could make a bad model look good. Cross-validation repeats the split k times, using each fold as test once, then averages. This gives a lower-variance, more reliable estimate of true generalisation performance — and the standard deviation tells you how stable the model is across different data subsets.
</details>

<details>
<summary><strong>Q2: What is data leakage?</strong></summary>

**Answer:** Data leakage is when information from the test set 'leaks' into the training process, causing overly optimistic evaluation. Common cases: (1) Fitting the scaler/imputer on the full dataset before splitting — test statistics contaminate training. (2) Using a future feature (e.g., 'was the loan repaid' as a feature for predicting if the loan was approved). Prevention: always fit preprocessing only on training data, then apply to test. Use sklearn Pipelines — they handle this correctly automatically.
</details>

<details>
<summary><strong>Q3: What is the difference between GridSearchCV and RandomizedSearchCV?</strong></summary>

**Answer:** GridSearchCV tries every combination of hyperparameters — guaranteed to find the best combination in the grid, but expensive (scales exponentially). RandomizedSearchCV samples n_iter random combinations from the parameter distributions — much faster, and empirically finds nearly-optimal parameters in a fraction of the time. Rule of thumb: use Grid for small search spaces (<50 combinations), Randomized for large spaces or continuous distributions.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 9 Complete!</h3>
<ul>
<li>k-fold and stratified k-fold cross-validation</li>
<li>GridSearchCV and RandomizedSearchCV</li>
<li>Learning curves for bias-variance diagnosis</li>
<li>Data leakage prevention</li>
</ul>
<p><strong>Next:</strong> Chapter 10 — Ensemble Methods</p>
</div>